<a href="https://colab.research.google.com/github/ahmedyomiisiaka/RP-Network-Analysis/blob/main/notebooks/Notebook_01_RP_Gene_Collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 01 — RP Gene Collection and Curation

**Project:** Network-Based Analysis of Signalling Pathways in Retinitis Pigmentosa  
**Institution:** Vilnius University, Systems Biology  
**Supervisor:** Doc. Dr. Erinija Pranckevičienė  
**Author:** Ahmed Yomi Isiaka  
**Stage:** 1 — RP Gene Collection  

---

## Objective

Collect and curate the complete list of genes associated with
Retinitis Pigmentosa (RP) from two independent sources:

1. **RetiGene database** (retigene.erdc.info) — the most current
   curated database of inherited retinal disease genes
2. **Rivolta et al. (2025), Table S1** — peer-reviewed publication
   reporting 103 confirmed non-syndromic RP genes

Both lists are cross-checked and reconciled into a single final
gene list that will be used as the source nodes in the PPI network.

## Input
- RetiGene API (live query)
- `mmc2.xlsx` — Rivolta et al. (2025) supplementary Table S1

## Output
| File | Description |
|------|-------------|
| `data/raw/rp_genes_retigene.csv` | All RP genes from RetiGene |
| `data/raw/rp_genes_rivolta2025.csv` | RP genes from Rivolta Table S1 |
| `data/processed/rp_genes_final.csv` | Final reconciled gene list |

## Reference
Rivolta C., et al. (2025). RetiGene: A comprehensive gene atlas
for inherited retinal diseases. *Am J Hum Genet*, 112(10):2253–2265.
https://doi.org/10.1016/j.ajhg.2025.08.017

In [9]:
# ─────────────────────────────────────────────────────────────────
# Notebook 01 — Cell 1: Install libraries and create folders
# ─────────────────────────────────────────────────────────────────

import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "requests", "pandas", "openpyxl"],
    check=True
)

import os
from pathlib import Path

# Create folder structure (same as Notebook 00)
DIRS = [
    "/content/rp_network/data/raw",
    "/content/rp_network/data/processed",
    "/content/rp_network/data/results",
    "/content/rp_network/docs",
    "/content/rp_network/figures",
]
for d in DIRS:
    Path(d).mkdir(parents=True, exist_ok=True)

print("✓ Libraries installed.")
print("✓ Folder structure ready.")

✓ Libraries installed.
✓ Folder structure ready.


In [11]:
# ─────────────────────────────────────────────────────────────────
# Notebook 01 — Cell 2: Import libraries
# ─────────────────────────────────────────────────────────────────

import requests
import pandas as pd
from datetime import datetime

print(f"pandas   : {pd.__version__}")
print(f"requests : {requests.__version__}")
print("✓ All imports successful.")

pandas   : 2.2.2
requests : 2.32.4
✓ All imports successful.


---
## Part A — RetiGene Database

RetiGene (https://retigene.erdc.info) is a Shiny web application —
it does not have a public REST API. Data must be downloaded manually.

**How to get the file:**
1. Go to https://retigene.erdc.info
2. Scroll to the Genes section
3. Click "Download Full Table"
4. Save the file to your computer

Then run the cell below to upload it to Colab.

**Current version:** v1.13 (curated until 13 July 2026)

**Citation:**
Rivolta C., et al. (2025). RetiGene: A comprehensive gene atlas
for inherited retinal diseases. *Am J Hum Genet*, 112(10):2253–2265.

In [18]:
# ─────────────────────────────────────────────────────────────────
# Notebook 01 — Cell 3: Upload RetiGene gene table
#
# Source  : RetiGene database v1.13 (https://retigene.erdc.info)
# Method  : Manual download from website → upload to Colab
#           (RetiGene is a Shiny app with no public API)
# File    : Downloaded using "Download Full Table" button
#
# Output  : data/raw/rp_genes_retigene.csv
# ─────────────────────────────────────────────────────────────────

from google.colab import files

print("Upload your RetiGene gene table CSV file.")
print("(Downloaded from https://retigene.erdc.info → Genes → Download Full Table)")
print()

uploaded = files.upload()

for filename in uploaded.keys():
    print(f"✓ Uploaded: {filename} ({len(uploaded[filename])} bytes)")
    retigene_filename = filename

Upload your RetiGene gene table CSV file.
(Downloaded from https://retigene.erdc.info → Genes → Download Full Table)



Saving genes.csv to genes.csv
✓ Uploaded: genes.csv (173390 bytes)


In [19]:
# ─────────────────────────────────────────────────────────────────
# Notebook 01 — Cell 4: Inspect RetiGene file structure
#
# First we look at what columns and values are in the file
# before applying any filter.
# ─────────────────────────────────────────────────────────────────

df_retigene_all = pd.read_csv(retigene_filename)

print(f"Total rows in RetiGene file : {len(df_retigene_all)}")
print()
print("Column names:")
for col in df_retigene_all.columns:
    print(f"  - {col}")
print()
print("First 3 rows:")
print(df_retigene_all.head(3).to_string())

Total rows in RetiGene file : 528

Column names:
  - Gene name
  - Other names
  - Year found
  - Broad phenotype category
  - Inheritance mode
  - Phenotype category (inheritance)
  - OMIM (04.02.2025)
  - Orphanet (03.12.2024)
  - Functional annotations
  - Pathogenic variant types
  - RNA-Seq tissues
  - scRNA-Seq broad
  - scRNA-Seq cells
  - References (PMIDs)

First 3 rows:
                    Gene name           Other names  Year found Broad phenotype category Inheritance mode Phenotype category (inheritance)                                                                                                                                                                                 OMIM (04.02.2025)                                                           Orphanet (03.12.2024)                                                                Functional annotations    Pathogenic variant types       RNA-Seq tissues scRNA-Seq broad    scRNA-Seq cells                   References (PMI

In [20]:
# ─────────────────────────────────────────────────────────────────
# Notebook 01 — Cell 5: Filter for RP genes from RetiGene
#
# After seeing the column names above, we identify:
#   - Which column contains the phenotype/disease name
#   - Which column contains the gene symbol
#
# We filter for rows where phenotype contains "RP" or
# "Retinitis Pigmentosa" (non-syndromic confirmed genes).
# ─────────────────────────────────────────────────────────────────

print("Unique values in phenotype column:")
print()

# Identify phenotype column — check common names
for possible_col in ["Broad phenotype category", "Phenotype category (inheritance)",
                     "phenotype", "disease", "Phenotype", "Disease"]:
    if possible_col in df_retigene_all.columns:
        print(f"Column found: '{possible_col}'")
        print(df_retigene_all[possible_col].value_counts().head(20).to_string())
        PHENOTYPE_COL = possible_col
        break

Unique values in phenotype column:

Column found: 'Broad phenotype category'
Broad phenotype category
Syndromic        301
Non-syndromic    162
Both              65


In [23]:
# ─────────────────────────────────────────────────────────────────
# Notebook 01 — Cell 6: Upload mmc2.xlsx (Rivolta Table S1)
#
# Source  : Rivolta et al. (2025), Am J Hum Genet, Table S1
# File    : mmc2.xlsx — supplementary file from the paper
#
# IMPORTANT: Colab loses uploaded files when the session resets.
# This cell saves the file to /content/rp_network/data/raw/
# so it persists within the session and is easy to find.
#
# If you see FileNotFoundError in the next cell, just re-run
# this cell to upload again.
# ─────────────────────────────────────────────────────────────────

from google.colab import files
import shutil

print("Click the button below and upload mmc2.xlsx")
print()

uploaded = files.upload()

for filename in uploaded.keys():
    # Save to project raw data folder
    dest = f"/content/rp_network/data/raw/{filename}"
    shutil.copy(filename, dest)
    print(f"✓ Uploaded and saved: {dest}")
    RIVOLTA_FILE = dest

Click the button below and upload mmc2.xlsx



Saving mmc2.xlsx to mmc2.xlsx
✓ Uploaded and saved: /content/rp_network/data/raw/mmc2.xlsx


In [24]:
# ─────────────────────────────────────────────────────────────────
# Notebook 01 — Cell 7: Read and filter Rivolta Table S1
#
# Source  : Rivolta et al. (2025), Am J Hum Genet, Table S1
# File    : mmc2.xlsx (uploaded in Cell 6)
#
# Filter applied:
#   - Retained = "Yes"     (confirmed genes only, not candidates)
#   - Category columns contain "RP"  (Retinitis Pigmentosa)
#   - Entry_Type = "Gene"  (exclude loci like "RP17 locus")
#
# Output  : data/raw/rp_genes_rivolta2025.csv
# ─────────────────────────────────────────────────────────────────

import openpyxl

# Use the path saved in Cell 6
# If this fails, re-run Cell 6 to re-upload the file
RIVOLTA_FILE = "/content/rp_network/data/raw/mmc2.xlsx"

print(f"Loading: {RIVOLTA_FILE}")

wb = openpyxl.load_workbook(RIVOLTA_FILE, read_only=True)
ws = wb.active

# Read all rows
rows = list(ws.iter_rows(values_only=True))

# Row 3 (index 2) is the header row
headers = list(rows[2])
print("Column headers found:")
for i, h in enumerate(headers):
    if h:
        print(f"  [{i}] {h}")

Loading: /content/rp_network/data/raw/mmc2.xlsx
Column headers found:
  [0] Gene
  [1] Year of first publication
  [2] Retained
  [3] Reason
  [4] Category
  [5] Inheritance
  [6] Multiple phenotypes, non-syndromic
  [7] Multiple phenotypes, syndromic
  [8] Category 1
  [9] Inheritance 1
  [10] Category 2
  [11] Inheritance 2
  [12] Category 3
  [13] Inheritance 3
  [14] Category 4
  [15] Inheritance 4
  [16] Category 5
  [17] Inheritance 5
  [18] PMIDs
  [19] Functional annotations
  [20] Annotation 1
  [21] Annotation 2
  [22] Annotation 3
  [23] Annotation 4
  [24] Annotation 5
  [25] Annotation 6
  [26] Pathogenic variants
  [27] RNA-Seq tissue
  [28] scRNA-Seq broad
  [29] scRNA-Seq specific
  [30] OMIM
  [31] Orphanet
  [32]  


In [25]:
# ─────────────────────────────────────────────────────────────────
# Notebook 01 — Cell 8: Extract RP genes from Rivolta Table S1
#
# Filter applied:
#   - Retained = "Yes"  (confirmed genes only)
#   - Any of columns 4,8,10,12,14,16 contains "RP"
#   - Excludes loci entries (e.g. "RP17 locus", "Xq27.1 locus")
#
# Output: data/raw/rp_genes_rivolta2025.csv
# ─────────────────────────────────────────────────────────────────

rp_entries = []

for row in rows[3:]:   # data starts at row index 3
    gene     = row[0]
    retained = row[2]

    if not gene or retained != "Yes":
        continue

    # Check all category columns for "RP"
    categories = [
        str(row[i]) if row[i] else ""
        for i in [4, 8, 10, 12, 14, 16]
    ]

    if any("RP" in c for c in categories):
        is_locus = "locus" in str(gene).lower()
        rp_entries.append({
            "gene_symbol"      : gene,
            "entry_type"       : "Locus" if is_locus else "Gene",
            "primary_category" : row[4],
            "inheritance"      : row[5],
            "rna_seq_tissue"   : row[27] if len(row) > 27 else "",
            "scrna_broad"      : row[28] if len(row) > 28 else "",
            "scrna_specific"   : row[29] if len(row) > 29 else "",
            "source"           : "Rivolta_2025_TableS1",
            "date_accessed"    : datetime.now().strftime("%Y-%m-%d"),
        })

df_rivolta = pd.DataFrame(rp_entries)
df_rivolta_genes = df_rivolta[df_rivolta["entry_type"] == "Gene"].copy()
df_rivolta_loci  = df_rivolta[df_rivolta["entry_type"] == "Locus"].copy()

print(f"Total RP entries from Rivolta 2025 : {len(df_rivolta)}")
print(f"  Genes : {len(df_rivolta_genes)}")
print(f"  Loci  : {len(df_rivolta_loci)}")
print(f"  Loci excluded: {df_rivolta_loci['gene_symbol'].tolist()}")
print()
print("Inheritance breakdown:")
print(df_rivolta_genes["inheritance"].value_counts().to_string())

# Save
out_path = "/content/rp_network/data/raw/rp_genes_rivolta2025.csv"
df_rivolta.to_csv(out_path, index=False)
print()
print(f"✓ Saved → {out_path}")

Total RP entries from Rivolta 2025 : 103
  Genes : 101
  Loci  : 2
  Loci excluded: ['RP17 locus', 'Xq27.1 locus']

Inheritance breakdown:
inheritance
AR          68
AD-AR       21
AD           8
X-linked     4

✓ Saved → /content/rp_network/data/raw/rp_genes_rivolta2025.csv


---
## Part C — Reconciliation

We now compare the two gene lists:

1. Genes in **both** sources → highest confidence
2. Genes **only in RetiGene** → newer additions since Rivolta 2025 publication
3. Genes **only in Rivolta** → may be missing from RetiGene filter

Primary source is **Rivolta 2025** (peer-reviewed, fixed 103-gene set).
RetiGene is used as cross-check and to capture newer additions.

In [26]:
# ─────────────────────────────────────────────────────────────────
# Notebook 01 — Cell 10: Cross-check both gene lists
#
# Compares:
#   - RetiGene v1.13 RP genes (114 genes, includes 2026 additions)
#   - Rivolta 2025 Table S1 RP genes (101 genes + 2 loci)
#
# Expected result:
#   - Most genes appear in both (high confidence)
#   - RetiGene has more genes = newer additions after paper publication
#   - A few may appear only in Rivolta = different filter logic
# ─────────────────────────────────────────────────────────────────

# Gene symbol sets — uppercase for comparison
retigene_genes = set(
    df_rp_retigene[GENE_COL].str.upper().dropna()
)

rivolta_genes = set(
    df_rivolta_genes["gene_symbol"].str.upper().dropna()
)

# Comparison
in_both       = retigene_genes & rivolta_genes
only_retigene = retigene_genes - rivolta_genes
only_rivolta  = rivolta_genes  - retigene_genes

print("=" * 55)
print("CROSS-CHECK RESULTS")
print("=" * 55)
print(f"RetiGene v1.13 RP genes  : {len(retigene_genes)}")
print(f"Rivolta 2025 RP genes    : {len(rivolta_genes)}")
print()
print(f"In BOTH sources          : {len(in_both)}")
print(f"Only in RetiGene         : {len(only_retigene)}")
print(f"Only in Rivolta 2025     : {len(only_rivolta)}")
print()

if only_retigene:
    print("Genes only in RetiGene (newer additions):")
    for g in sorted(only_retigene):
        print(f"  {g}")
print()

if only_rivolta:
    print("Genes only in Rivolta 2025:")
    for g in sorted(only_rivolta):
        print(f"  {g}")

CROSS-CHECK RESULTS
RetiGene v1.13 RP genes  : 114
Rivolta 2025 RP genes    : 101

In BOTH sources          : 101
Only in RetiGene         : 13
Only in Rivolta 2025     : 0

Genes only in RetiGene (newer additions):
  BCOR
  CREB3
  FSD1L
  PRPF6
  RNU4-2
  RNU6-1
  RNU6-2
  RNU6-8
  RNU6-9
  RP17 LOCUS
  SAXO6
  SCLT1
  XQ27.1 LOCUS



In [27]:
# ─────────────────────────────────────────────────────────────────
# Notebook 01 — Cell 11: Build final gene list
#
# Decision:
#   Primary source = Rivolta 2025 (101 genes, peer-reviewed)
#   + RetiGene-only genes added and flagged as "RetiGene only"
#     (these are newer additions since paper publication)
#   + 2 loci entries included but flagged as "Locus"
#
# Each entry annotated with:
#   gene_symbol      : HGNC gene symbol
#   entry_type       : Gene or Locus
#   inheritance      : AR, AD, AD-AR, X-linked
#   primary_category : RP category from Rivolta
#   in_rivolta_2025  : True/False
#   in_retigene      : True/False
#   confidence       : Both sources / Rivolta only / RetiGene only
#   source_primary   : which list is primary source
#   date_compiled    : today's date
#
# Output: data/processed/rp_genes_final.csv
# ─────────────────────────────────────────────────────────────────

final_rows = []

# --- Rivolta 2025 entries (primary source) ---
for _, row in df_rivolta.iterrows():
    symbol = str(row["gene_symbol"]).upper()
    in_rt  = symbol in retigene_genes

    final_rows.append({
        "gene_symbol"     : row["gene_symbol"],
        "entry_type"      : row["entry_type"],
        "inheritance"     : row["inheritance"],
        "primary_category": row["primary_category"],
        "rna_seq_tissue"  : row["rna_seq_tissue"],
        "scrna_broad"     : row["scrna_broad"],
        "in_rivolta_2025" : True,
        "in_retigene"     : in_rt,
        "confidence"      : "Both sources" if in_rt else "Rivolta 2025 only",
        "source_primary"  : "Rivolta_2025_TableS1",
        "date_compiled"   : datetime.now().strftime("%Y-%m-%d"),
    })

# --- RetiGene-only genes (newer additions) ---
for sym in sorted(only_retigene):
    # Get inheritance from RetiGene table
    rt_row = df_rp_retigene[
        df_rp_retigene[GENE_COL].str.upper() == sym
    ]
    inheritance = rt_row["Inheritance mode"].values[0] if len(rt_row) else ""
    phenotype   = rt_row[PHENOTYPE_COL].values[0] if len(rt_row) else ""

    final_rows.append({
        "gene_symbol"     : sym,
        "entry_type"      : "Gene",
        "inheritance"     : inheritance,
        "primary_category": phenotype,
        "rna_seq_tissue"  : "",
        "scrna_broad"     : "",
        "in_rivolta_2025" : False,
        "in_retigene"     : True,
        "confidence"      : "RetiGene only (post-2025 addition)",
        "source_primary"  : "RetiGene_v1.13",
        "date_compiled"   : datetime.now().strftime("%Y-%m-%d"),
    })

df_final = pd.DataFrame(final_rows)

# Summary
genes_only = df_final[df_final["entry_type"] == "Gene"]
loci_only  = df_final[df_final["entry_type"] == "Locus"]

print("=" * 55)
print("FINAL GENE LIST SUMMARY")
print("=" * 55)
print(f"Total entries            : {len(df_final)}")
print(f"  Genes                  : {len(genes_only)}")
print(f"  Loci                   : {len(loci_only)}")
print()
print("Confidence breakdown:")
print(df_final["confidence"].value_counts().to_string())
print()
print("Inheritance breakdown (genes only):")
print(genes_only["inheritance"].value_counts().to_string())

# Save
out_path = "/content/rp_network/data/processed/rp_genes_final.csv"
df_final.to_csv(out_path, index=False)
print()
print(f"✓ Final gene list saved → {out_path}")

FINAL GENE LIST SUMMARY
Total entries            : 116
  Genes                  : 114
  Loci                   : 2

Confidence breakdown:
confidence
Both sources                          103
RetiGene only (post-2025 addition)     13

Inheritance breakdown (genes only):
inheritance
AR          72
AD-AR       21
AD          15
X-linked     6

✓ Final gene list saved → /content/rp_network/data/processed/rp_genes_final.csv


In [28]:
# ─────────────────────────────────────────────────────────────────
# Notebook 01 — Cell 12: Update progress log
# ─────────────────────────────────────────────────────────────────

today = datetime.now().strftime("%Y-%m-%d")

log_update = f"""
---

## Entry 2 — {today}

### Objectives for today
- Collect RP-associated genes from RetiGene database
- Collect RP-associated genes from Rivolta et al. (2025) Table S1
- Cross-check both lists and produce final reconciled gene list

### What was done

**Part A — RetiGene v1.13:**
- Downloaded full gene table manually from https://retigene.erdc.info
- Total genes in database: 528 (all IRDs)
- Filtered for: Phenotype contains "RP" AND Broad category = Non-syndromic or Both
- RP genes retrieved: {len(retigene_genes)}

**Part B — Rivolta et al. 2025, Table S1:**
- Uploaded mmc2.xlsx (Am J Hum Genet, 2025)
- Filtered for: Retained = Yes, Category contains RP
- Genes: {len(rivolta_genes)} + 2 loci = 103 total entries

**Part C — Reconciliation:**
- Genes in both sources: {len(in_both)} (high confidence)
- Only in RetiGene: {len(only_retigene)} (newer additions since paper)
- Only in Rivolta: {len(only_rivolta)}
- Final gene list: {len(df_final)} total entries

### Output files
- data/raw/rp_genes_retigene.csv — RetiGene RP genes
- data/raw/rp_genes_rivolta2025.csv — Rivolta 2025 RP genes
- data/processed/rp_genes_final.csv — Final reconciled list

### Problems encountered
- RetiGene has no public API — data downloaded manually from website
- Colab session reset required re-uploading mmc2.xlsx

### Next steps (Stage 2 — Notebook 02)
- Collect apoptosis pathway proteins from Reactome R-HSA-109581
- Collect apoptosis pathway proteins from KEGG hsa04210
- Cross-check and produce final apoptosis target list

---
"""

log_path = "/content/rp_network/docs/progress_log.md"
with open(log_path, "a") as f:
    f.write(log_update)

print(f"✓ Progress log updated → {log_path}")

✓ Progress log updated → /content/rp_network/docs/progress_log.md


---
## Stage 1 Complete ✓

### Output files produced

| File | Contents |
|------|----------|
| `data/raw/rp_genes_retigene.csv` | 114 RP genes from RetiGene v1.13 |
| `data/raw/rp_genes_rivolta2025.csv` | 103 entries from Rivolta 2025 Table S1 |
| `data/processed/rp_genes_final.csv` | Final reconciled gene list |

### What to commit to GitHub

1. `data/processed/rp_genes_final.csv`
   Commit message: `Stage 1: final RP gene list`

2. Updated `docs/progress_log.md`
   Commit message: `Stage 1: update progress log`

3. This notebook:
   File → Save a copy in GitHub
   Path: `notebooks/Notebook_01_RP_Gene_Collection.ipynb`
   Commit message: `Add Notebook 01: RP gene collection`

### Next
Open a new notebook → **Notebook 02 — Apoptosis Pathway Gene Collection**

In [29]:
# ─────────────────────────────────────────────────────────────────
# Notebook 01 — Correction Cell: Fix loci classification
#
# RP17 locus and Xq27.1 locus appear in "RetiGene only" because
# the uppercase string comparison did not match the loci format
# between the two sources. These are present in both — correct it.
# ─────────────────────────────────────────────────────────────────

# Fix loci entries — mark them as "Both sources"
loci_mask = df_final["entry_type"] == "Locus"
df_final.loc[loci_mask, "in_rivolta_2025"] = True
df_final.loc[loci_mask, "in_retigene"]     = True
df_final.loc[loci_mask, "confidence"]      = "Both sources"
df_final.loc[loci_mask, "source_primary"]  = "Rivolta_2025_TableS1"

# Re-save
out_path = "/content/rp_network/data/processed/rp_genes_final.csv"
df_final.to_csv(out_path, index=False)

# Final clean summary
genes_only = df_final[df_final["entry_type"] == "Gene"]
loci_only  = df_final[df_final["entry_type"] == "Locus"]

print("=" * 55)
print("FINAL GENE LIST — CORRECTED SUMMARY")
print("=" * 55)
print(f"Total entries            : {len(df_final)}")
print(f"  Genes                  : {len(genes_only)}")
print(f"  Loci                   : {len(loci_only)}")
print()
print("Confidence breakdown:")
print(df_final["confidence"].value_counts().to_string())
print()
print("Inheritance breakdown (genes only):")
print(genes_only["inheritance"].value_counts().to_string())
print()
print(f"✓ Corrected file saved → {out_path}")

FINAL GENE LIST — CORRECTED SUMMARY
Total entries            : 116
  Genes                  : 114
  Loci                   : 2

Confidence breakdown:
confidence
Both sources                          103
RetiGene only (post-2025 addition)     13

Inheritance breakdown (genes only):
inheritance
AR          72
AD-AR       21
AD          15
X-linked     6

✓ Corrected file saved → /content/rp_network/data/processed/rp_genes_final.csv


In [31]:
# ─────────────────────────────────────────────────────────────────
# Download files for GitHub commit
# ─────────────────────────────────────────────────────────────────

from google.colab import files

print("Downloading files for GitHub commit...")
print()

to_download = [
    "/content/rp_network/data/processed/rp_genes_final.csv",
    "/content/rp_network/docs/progress_log.md",
]

for path in to_download:
    files.download(path)
    print(f"✓ Downloaded: {path.split('/')[-1]}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Downloaded: rp_genes_final.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Downloaded: progress_log.md
